# Species Classification via Decision Tree

Trains a decision tree classifier to predict species of origin from CGI
features, using two separate feature sets: (1) core CGI physical features
(GC content, length, obs/exp) and (2) 3-mer composition. Used to identify
which individual features drive species separability, complementing the
unsupervised DBSCAN results.

**Input**: `dataset/processed/cpg_islands.csv`, `dataset/processed/3mer_features.csv`

**Output**: printed classification report (best params, feature importances,
            tree rules) — not persisted to disk

In [ ]:
import pandas as pd
import numpy as np
import sys
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.tree import export_text

In [ ]:
# Load merged CGI dataset and 3-mer features.
# cgi_features: physical CGI features (columns 5 to second-last: gc-content, length, obs/exp)
# labels: species column (last column)
islands = pd.read_csv('../dataset/processed/cpg_islands.csv')
kmers = pd.read_csv('../dataset/processed/3mer_features.csv')

kmer_features = kmers.iloc[:, 1:] # drop 'id' column
cgi_features = islands.iloc[:, 5:-1] # columns 5:-1 = gc-content, length, obs/exp
labels = islands.iloc[:,-1:]

kmer_features['label'] = labels
cgi_features['label'] = labels

# Shuffle rows before train/test split
kmer_features =  kmer_features.sample(frac=1)
cgi_features = cgi_features.sample(frac=1)
train_sample_num = int(len(cgi_features)*0.8) # 80/20 train/test split

In [ ]:
d_tree = DecisionTreeClassifier()

params = {
    'max_depth': [3, 6, 9, 12], #max: log2(number of samples)
    'min_samples_split': [70, 210, 350], # 1%, 3%, 5%
    'criterion': ('gini', 'entropy')
}
# scoring='accuracy' may be sensitive to class imbalance across
# species (see README limitations)
clf = GridSearchCV(d_tree, params, scoring='accuracy')

In [ ]:
def result_print(clf, test, type, model, column):
  """
  Print a summary of GridSearchCV results: best hyperparameters,
  train/test scores, feature importances, and the resulting
  decision tree's split rules.

  Args:
      clf: fitted GridSearchCV object
      test (float): held-out test set accuracy
      type (str): label for the feature set being reported (e.g. "CGI")
      model: the best_estimator_ from clf
      column: feature column names, for labeling importances/rules
  """
  print(f'Result of Decision tree: for {type} features')
  print('\n Best parameters')
  parameters = clf.best_params_
  for p in parameters:
    print(f"    Best {p}: {parameters[p]}")
  print(f'\n Training score: {clf.best_score_}')
  print(f' Test score: {test}')
  print('\n Feature importances:')
  importances = model.feature_importances_
  feature_df = pd.DataFrame({'Feature': column, 'Importance': importances})
  print(feature_df.sort_values(by='Importance', ascending=False))
  print('\n  Tree dividing rules:')
  tree_rules = export_text(model, feature_names=list(column), max_depth=sys.maxsize)
  print(tree_rules)

In [ ]:
# Train on CGI physical features, evaluate on held-out test set
clf.fit(cgi_features.iloc[:train_sample_num,:-1],
        cgi_features['label'].iloc[:train_sample_num])
cgi_model = clf.best_estimator_
test_score = cgi_model.score(cgi_features.iloc[train_sample_num:,:-1],
        cgi_features['label'].iloc[train_sample_num:])

In [ ]:
result_print(clf, test_score, "CGI",
             cgi_model, cgi_features.iloc[:,:-1].columns)

Result of Decision tree: for CGI features

 Best parameters
    Best criterion: gini
    Best max_depth: 3
    Best min_samples_split: 350

 Training score: 0.6256459948320413
 Test score: 0.6242894056847546

 Feature importances:
      Feature  Importance
0  gc-content    0.898086
1      length    0.061000
2     obs/exp    0.040914

  Tree dividing rules:
|--- gc-content <= 0.57
|   |--- length <= 943.00
|   |   |--- obs/exp <= 0.73
|   |   |   |--- class: mustard
|   |   |--- obs/exp >  0.73
|   |   |   |--- class: mustard
|   |--- length >  943.00
|   |   |--- class: mustard
|--- gc-content >  0.57
|   |--- length <= 815.50
|   |   |--- class: mustard
|   |--- length >  815.50
|   |   |--- obs/exp <= 0.99
|   |   |   |--- class: mustard
|   |   |--- obs/exp >  0.99
|   |   |   |--- class: mustard



In [ ]:
# Train on 3-mer composition features (raw, not PCA-reduced — unlike clustering.ipynb)
clf.fit(kmer_features.iloc[:train_sample_num,:-1],
        kmer_features['label'].iloc[:train_sample_num])
kmer_model = clf.best_estimator_
test_score = kmer_model.score(kmer_features.iloc[train_sample_num:,:-1],
        kmer_features['label'].iloc[train_sample_num:])

In [ ]:
result_print(clf, test_score, "K-MER",
             kmer_model, kmer_features.iloc[:,:-1].columns)

Result of Decision tree: for K-MER features

 Best parameters
    Best criterion: gini
    Best max_depth: 3
    Best min_samples_split: 210

 Training score: 0.6247416020671834
 Test score: 0.627906976744186

 Feature importances:
   Feature  Importance
46     GCG    0.830525
59     CGC    0.099850
27     TGC    0.042615
1      AAT    0.019238
42     GGG    0.007771
..     ...         ...
58     CGG    0.000000
60     CCA    0.000000
61     CCT    0.000000
62     CCG    0.000000
63     CCC    0.000000

[64 rows x 2 columns]

  Tree dividing rules:
|--- GCG <= 0.04
|   |--- TGC <= 0.02
|   |   |--- AAT <= 0.01
|   |   |   |--- class: mustard
|   |   |--- AAT >  0.01
|   |   |   |--- class: mustard
|   |--- TGC >  0.02
|   |   |--- CGC <= 0.03
|   |   |   |--- class: mustard
|   |   |--- CGC >  0.03
|   |   |   |--- class: mustard
|--- GCG >  0.04
|   |--- CGC <= 0.03
|   |   |--- class: mustard
|   |--- CGC >  0.03
|   |   |--- GGG <= 0.00
|   |   |   |--- class: chinese_cabbage
|   | 